# 02b — MM-Fit rest and transition supplement

The first MM-Fit notebook exported only labeled exercise sets. A classifier trained only on exercises must still choose an exercise while the athlete rests, walks, or changes position. This notebook takes the gaps **outside every labeled set**, leaves a safety margin around the exercise boundaries, and saves short `unknown` pose clips.

Run this once and download `calorie_mmfit_unknown_v1.zip`. In notebook 03, upload it together with the Workout/Fitness and MM-Fit exercise ZIPs.


In [ ]:
from pathlib import Path
import csv
import hashlib
import os
import shutil
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from google.colab import files
from tqdm.auto import tqdm

MMFIT_URL = "https://s3.eu-west-2.amazonaws.com/vradu.uk/mm-fit.zip"
ARCHIVE_PATH = Path("/content/mm-fit.zip")
EXTRACT_ROOT = Path("/content/mm-fit-source")
OUTPUT_ROOT = Path("/content/calorie_mmfit_unknown_v1")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_FPS = 30
BOUNDARY_MARGIN_FRAMES = 30       # ignore one second around each exercise
UNKNOWN_CLIP_FRAMES = 120         # four-second clips
UNKNOWN_STRIDE_FRAMES = 120       # no overlapping copies
MAX_UNKNOWN_PER_WORKOUT = 12      # prevents unknown from dominating

print("Free temporary disk:")
os.system("df -h /content | tail -1")


In [ ]:
if not ARCHIVE_PATH.exists():
    print("Downloading the official MM-Fit archive...")
    urllib.request.urlretrieve(MMFIT_URL, ARCHIVE_PATH)
if not EXTRACT_ROOT.exists():
    print("Extracting MM-Fit...")
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(EXTRACT_ROOT)

pose_files = sorted(EXTRACT_ROOT.rglob("w*_pose_2d.npy"))
print("2D pose files:", len(pose_files))
assert pose_files, "Official archive layout changed or extraction failed."


## How the conversion works

For each workout we mark every frame belonging to any labeled exercise. We enlarge those regions by one second so that incomplete repetitions do not leak into `unknown`. The remaining gaps are divided into four-second clips. They keep the same person/workout `group_id`, which prevents one workout from appearing in both training and testing.


In [ ]:
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12


def label_path_for(pose_path):
    direct = pose_path.with_name(pose_path.name.replace("_pose_2d.npy", "_labels.csv"))
    if direct.exists():
        return direct
    workout_id = pose_path.name.split("_")[0]
    matches = list(pose_path.parent.rglob(f"{workout_id}_labels.csv"))
    if not matches:
        matches = list(EXTRACT_ROOT.rglob(f"{workout_id}_labels.csv"))
    return matches[0] if matches else None


def read_labels(path):
    with path.open(newline="") as handle:
        for row in csv.reader(handle):
            yield int(row[0]), int(row[1]), int(row[2]), row[3]


def unpack_pose_2d(array):
    if array.ndim != 3:
        raise ValueError(f"Unexpected pose rank: {array.shape}")
    if array.shape[0] == 2 and array.shape[2] >= 18:
        frame_numbers = array[0, :, 0].astype(int)
        xy = np.transpose(array[:, :, 1:18], (1, 2, 0))
    elif array.shape[-1] == 2 and array.shape[1] >= 17:
        frame_numbers = np.arange(array.shape[0])
        xy = array[:, :17, :]
    else:
        raise ValueError(f"Unsupported MM-Fit pose shape: {array.shape}")
    return frame_numbers, xy.astype(np.float32)


def normalize_2d(xy):
    result = []
    for frame in xy:
        hip_center = (frame[LEFT_HIP] + frame[RIGHT_HIP]) / 2.0
        shoulder_center = (frame[LEFT_SHOULDER] + frame[RIGHT_SHOULDER]) / 2.0
        shoulder_width = np.linalg.norm(frame[LEFT_SHOULDER] - frame[RIGHT_SHOULDER])
        torso_length = np.linalg.norm(shoulder_center - hip_center)
        scale = max(float(shoulder_width), float(torso_length), 1e-6)
        normalized_xy = (frame - hip_center) / scale
        z = np.zeros((17, 1), dtype=np.float32)
        visibility = np.ones((17, 1), dtype=np.float32)
        result.append(np.concatenate([normalized_xy, z, visibility], axis=1))
    return np.asarray(result, dtype=np.float32)


In [ ]:
rows = []
for pose_path in tqdm(pose_files, desc="Extracting unknown clips"):
    workout_id = pose_path.name.split("_")[0]
    labels_path = label_path_for(pose_path)
    if labels_path is None:
        continue

    frame_numbers, xy = unpack_pose_2d(np.load(pose_path))
    occupied = np.zeros(len(frame_numbers), dtype=bool)
    for start_frame, end_frame, _, _ in read_labels(labels_path):
        occupied |= (
            (frame_numbers >= start_frame - BOUNDARY_MARGIN_FRAMES)
            & (frame_numbers <= end_frame + BOUNDARY_MARGIN_FRAMES)
        )

    available = ~occupied
    candidates = []
    run_start = None
    for index, is_available in enumerate(np.r_[available, False]):
        if is_available and run_start is None:
            run_start = index
        elif not is_available and run_start is not None:
            run_end = index
            for start in range(run_start, run_end - UNKNOWN_CLIP_FRAMES + 1, UNKNOWN_STRIDE_FRAMES):
                candidates.append((start, start + UNKNOWN_CLIP_FRAMES))
            run_start = None

    # Spread the selected clips over the full workout instead of taking only the beginning.
    if len(candidates) > MAX_UNKNOWN_PER_WORKOUT:
        chosen = np.linspace(0, len(candidates) - 1, MAX_UNKNOWN_PER_WORKOUT).round().astype(int)
        candidates = [candidates[i] for i in sorted(set(chosen))]

    for clip_index, (start, end) in enumerate(candidates):
        landmarks = normalize_2d(xy[start:end])
        start_frame = int(frame_numbers[start])
        end_frame = int(frame_numbers[end - 1])
        clip_id = hashlib.sha1(
            f"{workout_id}:unknown:{start_frame}:{end_frame}".encode()
        ).hexdigest()[:16]
        output_path = OUTPUT_ROOT / "unknown" / f"{clip_id}.npz"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        timestamps_s = np.arange(len(landmarks), dtype=np.float32) / SOURCE_FPS
        np.savez_compressed(
            output_path,
            landmarks=landmarks,
            timestamps_s=timestamps_s,
            label="unknown",
            source_dataset="mm-fit",
            original_path=f"{workout_id}:unknown:{start_frame}-{end_frame}",
            target_fps=SOURCE_FPS,
            repetitions=0,
        )
        rows.append({
            "clip_id": clip_id,
            "source_dataset": "mm-fit",
            "person_id": workout_id,
            "group_id": f"mm-fit:{workout_id}",
            "source_class": "non_activity",
            "canonical_class": "unknown",
            "original_path": f"{pose_path.relative_to(EXTRACT_ROOT)}:{start_frame}-{end_frame}",
            "output_path": str(output_path.relative_to(OUTPUT_ROOT)),
            "status": "ok",
            "source_fps": SOURCE_FPS,
            "sampled_frames": len(landmarks),
            "detection_rate": 1.0,
            "duration_s": len(landmarks) / SOURCE_FPS,
            "repetitions": 0,
        })

manifest = pd.DataFrame(rows)
assert len(manifest) > 0, "No safe unknown intervals were found."
manifest.to_csv(OUTPUT_ROOT / "manifest.csv", index=False)
print("Unknown clips:", len(manifest))
print("Workouts:", manifest["person_id"].nunique())
print(manifest.groupby("person_id").size().describe().round(2))
display(manifest.head())


In [ ]:
npz_files = list(OUTPUT_ROOT.rglob("*.npz"))
assert len(npz_files) == len(manifest)
assert manifest["person_id"].nunique() >= 15
assert set(manifest["canonical_class"]) == {"unknown"}

zip_path = shutil.make_archive(
    "/content/calorie_mmfit_unknown_v1", "zip", root_dir=OUTPUT_ROOT
)
print("ZIP ready:", zip_path)
print("ZIP size MB:", round(Path(zip_path).stat().st_size / 1024**2, 2))
files.download(zip_path)
